# Controlled discharge: SoC estimator characterization

Runs the four SoC methods on one battery's controlled discharge test and produces the report's characterization tables and figures.
Set `BATTERY` in the next cell and run all cells. Run it once for LiNMC and once for LiFePO4.

**How to read the results.** This is a descriptive characterization, not an accuracy ranking.
- Each battery model was fitted to this same run, so the run is in-sample.
- The comparison reference is a *characterization SoC* built from the measured current (Table 2). It is not ground truth.
- At the rest points the model was fitted to, agreement from coulomb counting and OCV mapping is an implementation check, not evidence of accuracy.

**Methods.** All come from `soc_eval.py`, with the frozen settings.
- **Coulomb counting (CC), EKF and EKF + KCL** integrate the offset-corrected battery current: each sensor's zero offset is learned while that sensor is idle. Offset correction is a shared input step, not a method. Table 6 shows its effect on its own.
- **OCV mapping** looks SoC up from the terminal voltage at every step.
- **EKF + KCL** adds the gated current-balance (KCL) update. `kcl_correction_explained.md` concludes that this update is harmful as a corrector.
- Every method starts from the characterization SoC at t = 0.

Outputs go to `final_report/figures/` and `final_report/tables/`, with the prefix `<battery>_controlled`.
Run time: under a minute for LiNMC, about 15–20 minutes for LiFePO4 (1.3 million median steps, plus the before-tuning run).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'final_report' / 'soc_eval.py').exists()) / 'final_report'))
import soc_eval as se

BATTERY = 'LiNMC'                          # 'LiNMC' or 'LiFePO4'
START_OFFSETS = (-0.2, -0.1, 0.1, 0.2)     # initialization spread: extra starts relative to t = 0 (clipped to 0-100 %)
RUN_BEFORE_TUNING = True                   # also run the original server settings (Appendix C)

se.apply_style()
pd.set_option('display.precision', 3)
pd.set_option('display.width', 200)
cfg = se.BATTERIES[BATTERY]
PREFIX = f'{BATTERY}_controlled'
TITLE = f'{BATTERY} controlled discharge'

## Load the run and run the methods
Rests are blocks where every current is below 0.5 A for at least 5 minutes.
For LiNMC, the 8 Apr session follows a week-long rest. It is plotted but left out of every metric, as in the tuning.

In [2]:
model = se.load_model(cfg['model'])
src = cfg['controlled']
win = se.window_median(se.load_run(se.TEST_DATA / src['folder'], src['slot']))
rests = se.find_rests(win)
sessions = cfg['score_sessions']
scored_rests = rests if sessions is None else rests[rests.session.isin(sessions)]

ref_all = se.characterization_soc(win, BATTERY, model)
soc0 = ref_all[0]
settings = se.settings(BATTERY, model, 'tuned')
est, extra = se.run_methods(win, model, settings, slot=src['slot'], soc0=soc0, start_offsets=START_OFFSETS)
est = se.add_reference(est, win, BATTERY, model)
print(f"{model['name']}: Q_total {model['Q_total']:.2f} Ah. {len(win):,} median steps, {win.t.iloc[-1] / 3600:.2f} h recorded, "
      f"{len(rests)} rests ({len(scored_rests)} in scored sessions). Start: characterization SoC {100 * soc0:.2f} %.")

LiNMC: Q_total 71.43 Ah. 62,310 median steps, 8.68 h recorded, 13 rests (13 in scored sessions). Start: characterization SoC 99.53 %.


## Table 1: test completeness

In [3]:
completeness = se.completeness_table(win, rests, BATTERY)
se.save_table(completeness, f'{PREFIX}_completeness')
display(completeness)

,LiNMC
sessions,5
recorded time (h),8.679
"sessions: start of the recording file, recorded length","0: 18 Mar 16:00, 2.01 h; 1: 25 Mar 14:43, 2.96..."
discharge steps (battery |I| > 1 A for 30 s or more),14
step current range (A),26.2 to 31.1
"rests (all currents < 0.5 A, at least 5 min)",13
rest length (min): min / median / max,17 / 31 / 36
rests where 10 mV/min was reached,13 of 13
"time below the cutoff voltage (h), excluded",0.069
sessions excluded from scoring (numbered from 0),"3, 4"


## Table 2: how the characterization SoC is built
The characterization SoC integrates the measured current. Its uncertainty is therefore set by how much the two current sensors disagree and by their readings at rest.
- The rest-point table gives the voltage and the characterization SoC at each rest.
- The figure places the rest-end voltages on the model's OCV curve. This is in-sample: the model was fitted to these rests.

In [4]:
reference = se.reference_table(win, rests, BATTERY, model)
points = se.rest_points(est, rests)
se.save_table(reference, f'{PREFIX}_reference')
se.save_table(points, f'{PREFIX}_rest_points')
display(reference)
display(points[['session', 'type', 'length_min', 'ref_soc_t0_pct', 'ref_soc_end_pct', 'V_t0', 'V_end']])
fig = se.plot_rest_ocv(points, model, TITLE)
se.save(fig, f'{PREFIX}_rest_ocv')
plt.show()

,LiNMC
current used,mean of battery and load sensor
anchor,0 % at the last sample above the cutoff
"Q_total (Ah, from the model)",71.426
"Ah, battery sensor",69.843
"Ah, load sensor",72.336
"Ah, mean of the two",71.089
"sensor disagreement (load vs battery, %)",3.506
battery sensor reading at rest (A): first / last rest,-0.170 / -0.094
load sensor reading at rest (A): min / max,-0.052 / +0.067


,session,type,length_min,ref_soc_t0_pct,ref_soc_end_pct,V_t0,V_end
rest,,,,,,,
1,0,after_discharge,16.960,91.724,91.759,56.592,57.050
2,0,after_discharge,36.272,84.380,84.473,55.739,56.333
3,0,after_discharge,32.852,75.966,76.044,54.667,55.395
4,1,after_discharge,31.553,68.375,68.426,53.553,54.103
5,1,after_discharge,20.068,60.943,60.978,52.727,53.123
6,1,after_discharge,16.788,54.047,54.073,51.983,52.382
7,1,after_discharge,21.128,47.087,47.119,51.352,51.801
8,1,after_discharge,31.279,39.235,39.282,50.792,51.326
9,2,after_discharge,28.149,31.819,31.872,50.263,50.840


C:\Users\Kor\AppData\Local\Temp\ipykernel_29072\3155699393.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure: whole run and one rest
- **Panels:** battery current, voltage, and every method's SoC against the characterization SoC (dashed).
- **Markers:** rests are shaded. t₀ (open dot) is 10 s after the rest starts. t₁ (filled dot) is when the voltage settles to 10 mV/min.
- **Close-up:** a representative rest near 50% SoC.

In [5]:
fig = se.plot_whole_run(est, rests, TITLE)
se.save(fig, f'{PREFIX}_whole_run')
rest_id = se.representative_rest(points.loc[scored_rests.rest])
fig = se.plot_rest_closeup(est, rests, points, rest_id, TITLE)
se.save(fig, f'{PREFIX}_rest_{rest_id}_closeup')
plt.show()

C:\Users\Kor\AppData\Local\Temp\ipykernel_29072\2686317468.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Table 3 and figure: difference from the characterization SoC
d = method SoC − characterization SoC. It is measured two ways:
- at the end of each rest;
- over the scored window: LiNMC 3.4–92% SoC with the 8 Apr session excluded, LiFePO4 11–84%.

Call it the *difference from the characterization estimate*, not an error. For CC and OCV mapping at the fitted rest points it is an implementation check.

In [6]:
div = se.divergence(est, scored_rests)
scores = se.score_table(est, scored_rests, extra)
se.save_table(div, f'{PREFIX}_divergence', index=False)
se.save_table(scores, f'{PREFIX}_scores')
display(scores)
display(div.pivot(index='rest', columns='method', values='d_pp')[list(se.METHODS.values())].round(3))
fig = se.plot_divergence(div, points.loc[scored_rests.rest], TITLE)
se.save(fig, f'{PREFIX}_divergence')
plt.show()

,RMS d (pp),max |d| (pp),d at end of scored data (pp),median |d| at rest ends (pp),max |d| at rest ends (pp),max |ΔSoC| during a rest (pp),voltage innovation RMS (mV)
method,,,,,,,
Coulomb counting,0.065,0.156,-0.144,0.057,0.156,0.000,NaN
OCV mapping,9.335,25.024,-3.406,0.131,0.429,8.145,NaN
EKF,0.248,0.597,0.115,0.070,0.542,0.068,35.978
EKF + KCL,0.250,0.597,0.087,0.071,0.542,0.059,36.077


method,Coulomb counting,OCV mapping,EKF,EKF + KCL
rest,,,,
1,0.039,0.429,0.542,0.542
2,-0.022,0.342,0.461,0.461
3,-0.076,0.190,0.301,0.289
4,-0.066,0.217,0.038,0.027
5,-0.048,0.170,0.070,0.061
6,-0.028,0.139,0.078,0.071
7,-0.011,0.089,0.055,0.049
8,-0.004,0.064,0.039,0.032
9,-0.057,0.014,0.006,-0.010


C:\Users\Kor\AppData\Local\Temp\ipykernel_29072\2495419042.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Table 4 and figure: what each method does during a rest
During a rest the current is zero, so a consistent method should hold SoC still.
The figure shows the signed ΔSoC over each rest (t₀ → end), the rest's voltage changes, and the time taken to reach 10 mV/min.

In [7]:
deltas = se.rest_deltas(est, scored_rests)
rest_summary = se.rest_delta_table(deltas)
se.save_table(deltas, f'{PREFIX}_rest_deltas', index=False)
se.save_table(rest_summary, f'{PREFIX}_rest_summary')
display(rest_summary)
fig = se.plot_rest_summary(deltas, points.loc[scored_rests.rest], TITLE)
se.save(fig, f'{PREFIX}_rest_summary')
plt.show()

,rests,median |ΔSoC| t0→end (pp),max |ΔSoC| t0→end (pp),median |ΔSoC| t0→t1 (pp),max |ΔSoC| t0→t1 (pp)
method,,,,,
Coulomb counting,13,0.000,0.000,0.000,0.000
OCV mapping,13,4.579,8.145,4.127,7.092
EKF,13,0.005,0.068,0.003,0.063
EKF + KCL,13,0.010,0.059,0.003,0.059


C:\Users\Kor\AppData\Local\Temp\ipykernel_29072\402504150.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Table 5: shortened rest
Could a rest stop at t₁? For each rest, the table gives:
- the time from t₀ to t₁;
- the voltage change after t₁;
- how far each method's SoC moves between t₁ and the end of the rest.

In [8]:
after_t1 = deltas.assign(dsoc_t1_end_pp=deltas.dsoc_t0_end_pp - deltas.dsoc_t0_t1_pp)
shortened = points.loc[scored_rests.rest, ['length_min', 't1_minus_t0_min', 'criterion_met', 'dV_t0_t1_mV', 'dV_t1_end_mV']].join(
    after_t1.pivot(index='rest', columns='method', values='dsoc_t1_end_pp')[list(se.METHODS.values())]
    .add_prefix('ΔSoC t₁→end (pp): '))
shortened = shortened.round(4)
se.save_table(shortened, f'{PREFIX}_shortened_rest')
display(shortened)

,length_min,t1_minus_t0_min,criterion_met,dV_t0_t1_mV,dV_t1_end_mV,ΔSoC t₁→end (pp): Coulomb counting,ΔSoC t₁→end (pp): OCV mapping,ΔSoC t₁→end (pp): EKF,ΔSoC t₁→end (pp): EKF + KCL
rest,,,,,,,,,
1,16.960,5.855,True,421.055,37.344,0.0,0.325,-1.420e-02,-1.420e-02
2,36.272,6.939,True,506.406,88.125,0.0,0.866,-5.000e-03,-5.000e-03
3,32.852,9.575,True,663.250,64.406,0.0,0.459,-1.780e-02,-1.510e-02
4,31.553,8.064,True,424.219,125.859,0.0,0.865,1.470e-02,1.510e-02
5,20.068,5.433,True,348.188,47.812,0.0,0.409,1.360e-02,1.470e-02
6,16.788,5.362,True,357.188,42.000,0.0,0.452,6.800e-03,7.300e-03
7,21.128,6.362,True,400.266,48.562,0.0,0.713,8.000e-04,1.100e-03
8,31.279,9.336,True,468.188,66.047,0.0,1.053,7.000e-04,9.000e-04
9,28.149,8.976,True,479.578,97.422,0.0,1.390,4.200e-03,4.600e-03


## Table 6: input current, raw vs offset-corrected
The same CC and EKF, fed either the raw battery current or the offset-corrected current.
This shows the effect of offset correction on its own. It is not credited to any method.

In [9]:
inputs = se.score_table(est, scored_rests, extra, methods={
    'soc_cc_raw': 'CC, raw current', 'soc_cc': 'CC, offset-corrected',
    'soc_ekf_raw': 'EKF, raw current', 'soc_ekf': 'EKF, offset-corrected'})
se.save_table(inputs, f'{PREFIX}_input_current')
display(inputs)

,RMS d (pp),max |d| (pp),d at end of scored data (pp),median |d| at rest ends (pp),max |d| at rest ends (pp),max |ΔSoC| during a rest (pp),voltage innovation RMS (mV)
method,,,,,,,
"CC, raw current",0.989,1.663,1.663,0.962,1.723,0.144,NaN
"CC, offset-corrected",0.065,0.156,-0.144,0.057,0.156,0.000,NaN
"EKF, raw current",0.625,0.817,0.483,0.581,0.799,0.161,34.794
"EKF, offset-corrected",0.248,0.597,0.115,0.070,0.542,0.068,35.978


## Appendix A: initialization spread
Each method is replayed from several starting SoCs (clipped to 0–100%), and the spread is max − min across those starts.
- CC keeps its starting spread for the whole run.
- OCV mapping has no memory, so its spread is zero by construction.
- The EKF should shrink the spread wherever the voltage carries SoC information. On the LiFePO4 plateau it carries almost none.

In [10]:
sp = se.spread(extra)
h = est.t.to_numpy() / 3600
k1 = min(np.searchsorted(h, 1.0), len(h) - 1)
k_end = np.flatnonzero(est.V > se.V_CUTOFF)[-1]
first_below = lambda x: h[np.flatnonzero(x < 1.0)[0]] if (x < 1.0).any() else np.nan
spread_table = pd.DataFrame({se.METHODS[c]: {'at start (pp)': sp[c][0], 'after 1 h (pp)': sp[c][k1], 'at end (pp)': sp[c][k_end],
                                              'first below 1 pp (h)': first_below(sp[c])} for c in se.METHODS}).T
spread_table = spread_table.round(4)
se.save_table(spread_table, f'{PREFIX}_init_spread')
print('Starts (%):', np.round(100 * extra['starts'], 1))
display(spread_table)
fig = se.plot_spread(est, extra, TITLE)
se.save(fig, f'{PREFIX}_init_spread')
plt.show()

Starts (%): [ 99.5  79.5  89.5 100.  100. ]


,at start (pp),after 1 h (pp),at end (pp),first below 1 pp (h)
Coulomb counting,20.471,20.471,20.471,NaN
OCV mapping,0.000,0.000,0.000,1.000e-04
EKF,0.008,0.000,0.000,1.000e-04
EKF + KCL,0.008,0.000,0.000,1.000e-04


C:\Users\Kor\AppData\Local\Temp\ipykernel_29072\1721673714.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Appendix B: current corrections
- **Battery bias** learned by offset correction and by the gated KCL corrector. Dots are the raw readings during each rest.
- **KCL residual** (load − MPPT − batteries) before and after correction.
- **EKF + KCL minus EKF**, which isolates the effect of the balance update.

In [11]:
fig = se.plot_kcl(est, extra, scored_rests, TITLE, src['slot'])
se.save(fig, f'{PREFIX}_kcl')
ch = 2 if src['slot'] == 'main' else 3
res = {'raw': extra['res_raw'], 'after offsets': extra['res_offsets'], 'after gated KCL': extra['res_kcl']}
kcl_table = pd.DataFrame({
    'battery charge out (Ah)': [extra['Ah']['raw'], extra['Ah']['offsets'], extra['Ah']['kcl']],
    'median |KCL residual| (A)': [np.median(np.abs(r)) for r in res.values()],
    'battery bias at end (A)': [0.0, extra['bias_offsets'][ch, -1], extra['bias_kcl'][ch, -1]],
}, index=list(res))
se.save_table(kcl_table, f'{PREFIX}_kcl')
display(kcl_table)
diff = (100 * (est.soc_ekf_kcl - est.soc_ekf)).groupby((est.t // 60).astype(int)).median()
print(f"EKF + KCL minus EKF: largest per-minute median {diff.abs().max():.3f} pp, at the end {diff.iloc[-1]:+.3f} pp")
plt.show()

,battery charge out (Ah),median |KCL residual| (A),battery bias at end (A)
raw,69.843,0.207,0.000
after offsets,71.193,0.010,-0.098
after gated KCL,71.280,0.010,-0.170


EKF + KCL minus EKF: largest per-minute median 0.065 pp, at the end +0.000 pp


C:\Users\Kor\AppData\Local\Temp\ipykernel_29072\887683514.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Appendix C: before tuning
The same metrics with the original server settings. Start and rests are the same as above. The original settings are:
- process noise from `noise.json`;
- P0 = 1e-8;
- R = σ_v² per sample;
- raw current for CC and the EKF;
- the repo KCL corrector for EKF + KCL.

In [12]:
if RUN_BEFORE_TUNING:
    est_o, extra_o = se.run_methods(win, model, se.settings(BATTERY, model, 'original'), slot=src['slot'], soc0=soc0)
    est_o = se.add_reference(est_o, win, BATTERY, model)
    before = pd.concat({'original': se.score_table(est_o, scored_rests, extra_o), 'tuned': scores}, names=['settings', 'method'])
    before = before.reorder_levels(['method', 'settings']).loc[list(se.METHODS.values())]
    se.save_table(before, f'{PREFIX}_before_tuning')
    print(f"Original repo corrector reverted to raw currents on {100 * extra_o['reverted_frac']:.1f} % of steps")
    display(before)

Original repo corrector reverted to raw currents on 59.0 % of steps


RMS d (pp)  max |d| (pp)  d at end of scored data (pp)  median |d| at rest ends (pp)  max |d| at rest ends (pp)  max |ΔSoC| during a rest (pp)  voltage innovation RMS (mV)
method           settings                                                                                                                                                                             
Coulomb counting original       0.989         1.663                         1.663                         0.962                      1.723                          0.144                          NaN
                 tuned          0.065         0.156                        -0.144                         0.057                      0.156                          0.000                          NaN
OCV mapping      original       9.335        25.024                        -3.406                         0.131                      0.429                          8.145                          NaN
                 tuned          9.335        25.024                        -3.406                         0.131                      0.429                          8.145                          NaN
EKF              original       0.674         0.902                         0.491                         0.606                      0.883                          0.150                       12.537
                 tuned          0.248         0.597                         0.115                         0.070                      0.542                          0.068                       35.978
EKF + KCL        original       1.292         2.194                         1.023                         1.006                      2.189                          0.190                       11.695
                 tuned          0.250         0.597                         0.087                         0.071                      0.542                          0.059                       36.077